# 5. Klasifikasi Status Inflasi Bulan Depan
Memprediksi status inflasi tahunan indeks pangan bulan depan per pasar: Normal (< 5%), Waspada (5% sampai < 10%), Lonjakan (>= 10%). Tiga model dibandingkan dengan tebakan sederhana.

In [ ]:
import re

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, f1_score, silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.tree import DecisionTreeClassifier

import os
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

data_path = '../data/processed/data_bersih.csv'
cuaca_path = '../data/external/cuaca_bulanan.csv'
oni_path = '../data/external/oni.csv'
output_dir = '../data/processed'

## Konfigurasi dan fungsi

In [ ]:
KOL_PASAR = 'mkt_name'
KOL_TANGGAL = 'date'
KOL_INDEKS = 'food_price_index'
KOMODITAS = ['eggs', 'garlic', 'meat_beef', 'meat_chicken', 'meat_chicken_broiler', 'oil', 'onions', 'rice', 'sugar']

BATAS_WASPADA = 5
BATAS_LONJAKAN = 10
NAMA_KELAS = ['Normal', 'Waspada', 'Lonjakan']
FITUR_DASAR = ['inflasi_ini', 'inflasi_lalu', 'bulan']
PERSEN_TEST = 0.2
TAHUN_ACUAN_AKHIR = 2019

BOBOT_RISIKO = 0.6
BOBOT_MOMENTUM = 0.2
BOBOT_VOLATILITAS = 0.2
BATAS_PRIORITAS = 50
BATAS_PANTAU = 25
BATAS_HARGA_PERSEN = 5

NAMA_KOMODITAS = {
    'eggs': 'Telur', 'garlic': 'Bawang Putih', 'meat_beef': 'Daging Sapi', 'meat_chicken': 'Daging Ayam',
    'meat_chicken_broiler': 'Daging Ayam Broiler', 'oil': 'Minyak Goreng', 'onions': 'Bawang Merah',
    'rice': 'Beras', 'sugar': 'Gula', 'food_price_index': 'Indeks Pangan',
}
MUSIM_ONI = ['DJF', 'JFM', 'FMA', 'MAM', 'AMJ', 'MJJ', 'JJA', 'JAS', 'ASO', 'SON', 'OND', 'NDJ']

In [ ]:
def status_inflasi(inflasi):
    return np.select([inflasi < BATAS_WASPADA, inflasi < BATAS_LONJAKAN], [0, 1], default=2)


def tambah_rentang(d):
    h, l, c = f'h_{KOL_INDEKS}', f'l_{KOL_INDEKS}', f'c_{KOL_INDEKS}'
    if {h, l, c}.issubset(d.columns):
        return (d[h] - d[l]) / d[c] * 100
    return pd.Series(np.nan, index=d.index)


def siapkan_fitur(df):
    kol_inf = f'inflation_{KOL_INDEKS}'
    d = df.sort_values([KOL_PASAR, KOL_TANGGAL]).reset_index(drop=True)
    g = d.groupby(KOL_PASAR)
    out = pd.DataFrame({KOL_PASAR: d[KOL_PASAR], KOL_TANGGAL: d[KOL_TANGGAL]})
    out['inflasi_ini'] = d[kol_inf]
    out['inflasi_lalu'] = g[kol_inf].shift(1)
    out['inflasi_depan'] = g[kol_inf].shift(-1)
    out['bulan'] = d[KOL_TANGGAL].dt.month
    for k in KOMODITAS:
        if f'inflation_{k}' in d.columns:
            out[f'inf_{k}'] = d[f'inflation_{k}']
    if f'trust_{KOL_INDEKS}' in d.columns:
        out['trust_indeks'] = d[f'trust_{KOL_INDEKS}']
    for k in KOMODITAS:
        if f'trust_{k}' in d.columns:
            out[f'trust_{k}'] = d[f'trust_{k}']
    rentang = tambah_rentang(d)
    if rentang.notna().any():
        out['rentang_harga'] = rentang
    if 'spatially_interpolated' in d.columns:
        out['interpolasi'] = pd.to_numeric(d['spatially_interpolated'], errors='coerce')
    mentah = [k for k in KOMODITAS if k in d.columns]
    if mentah:
        ada_survei = (d[mentah].notna().sum(axis=1) > 0).astype(float)
        out['obs_12b'] = ada_survei.groupby(d[KOL_PASAR]).transform(lambda x: x.rolling(12, min_periods=1).sum())
    periode = d[KOL_TANGGAL].dt.to_period('M')
    out['lalu_ok'] = g[KOL_TANGGAL].shift(1).dt.to_period('M') == periode - 1
    out['depan_ok'] = g[KOL_TANGGAL].shift(-1).dt.to_period('M') == periode + 1
    return out


def daftar_fitur_kaya(out):
    tambahan = [c for c in out.columns if c.startswith(('inf_', 'trust_')) or c in ('rentang_harga', 'interpolasi')]
    return FITUR_DASAR + tambahan


def siapkan_data_model(out):
    dm = out[out['lalu_ok'] & out['depan_ok']].dropna(subset=['inflasi_ini', 'inflasi_lalu', 'inflasi_depan']).copy()
    dm['kelas_depan'] = status_inflasi(dm['inflasi_depan'].values)
    return dm


def pisah_waktu(dm, persen_test=PERSEN_TEST):
    bulan_unik = np.sort(dm[KOL_TANGGAL].unique())
    tgl_potong = pd.Timestamp(bulan_unik[int(len(bulan_unik) * (1 - persen_test))])
    return dm[KOL_TANGGAL] < tgl_potong, tgl_potong


def buat_model(jenis):
    if jenis == 'tree':
        est = DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=42)
    else:
        est = RandomForestClassifier(n_estimators=150, max_depth=8, min_samples_leaf=20,
                                     class_weight='balanced_subsample', n_jobs=-1, random_state=42)
    return make_pipeline(SimpleImputer(strategy='median'), est)


def bandingkan_model(dm, is_train, fitur_kaya, fitur_eksternal=None):
    y_train, y_test = dm.loc[is_train, 'kelas_depan'], dm.loc[~is_train, 'kelas_depan']
    skenario = {
        'Decision Tree (3 fitur)': ('tree', FITUR_DASAR),
        'Decision Tree (diperkaya)': ('tree', fitur_kaya),
        'Random Forest (diperkaya)': ('forest', fitur_kaya),
    }
    if fitur_eksternal:
        skenario['Random Forest + data eksternal'] = ('forest', fitur_kaya + fitur_eksternal)
    model, pred, baris = {}, {}, []
    for nama, (jenis, fitur) in skenario.items():
        model[nama] = buat_model(jenis).fit(dm.loc[is_train, fitur], y_train)
        pred[nama] = model[nama].predict(dm.loc[~is_train, fitur])
        baris.append((nama, accuracy_score(y_test, pred[nama]), f1_score(y_test, pred[nama], average='macro', zero_division=0)))
    pred_base = status_inflasi(dm.loc[~is_train, 'inflasi_ini'].values)
    baris.append(('Tebakan sederhana', accuracy_score(y_test, pred_base), f1_score(y_test, pred_base, average='macro', zero_division=0)))
    pred_normal = np.zeros(len(y_test), dtype=int)
    baris.append(('Selalu tebak Normal', accuracy_score(y_test, pred_normal), f1_score(y_test, pred_normal, average='macro', zero_division=0)))
    metrik = pd.DataFrame(baris, columns=['Model', 'Akurasi', 'F1 (macro)']).set_index('Model')
    terbaik = metrik.drop(index=['Tebakan sederhana', 'Selalu tebak Normal'])['F1 (macro)'].idxmax()
    return {'model': model, 'pred': pred, 'metrik': metrik, 'terbaik': terbaik, 'skenario': skenario, 'y_test': y_test}


def prediksi_terbaru(out, dm, hasil, fitur_kaya):
    jenis, fitur = hasil['skenario'][hasil['terbaik']]
    final = buat_model(jenis).fit(dm[fitur], dm['kelas_depan'])
    terbaru = out[(out[KOL_TANGGAL] == out[KOL_TANGGAL].max()) & out['lalu_ok']].dropna(subset=['inflasi_ini', 'inflasi_lalu']).copy()
    proba = pd.DataFrame(final.predict_proba(terbaru[fitur]), index=terbaru.index, columns=final.classes_)
    for k in range(3):
        terbaru[f'p_{NAMA_KELAS[k].lower()}'] = proba[k] if k in proba.columns else 0.0
    terbaru['status_bulan_ini'] = [NAMA_KELAS[i] for i in status_inflasi(terbaru['inflasi_ini'].values)]
    terbaru['status_prediksi_bulan_depan'] = [NAMA_KELAS[i] for i in final.predict(terbaru[fitur])]
    return terbaru, final, fitur


def uji_peringkat(dm, is_train, hasil, persen=0.1):
    jenis, fitur = hasil['skenario'][hasil['terbaik']]
    proba = hasil['model'][hasil['terbaik']].predict_proba(dm.loc[~is_train, fitur])
    kelas = list(hasil['model'][hasil['terbaik']].classes_)
    risiko = proba[:, kelas.index(1)] * 0.5 + proba[:, kelas.index(2)] if 2 in kelas else proba[:, kelas.index(1)] * 0.5
    lonjakan = (hasil['y_test'].values == 2)
    n_teratas = max(int(len(risiko) * persen), 1)
    teratas = np.argsort(-risiko)[:n_teratas]
    return {'persen': persen, 'tertangkap': float(lonjakan[teratas].sum() / max(lonjakan.sum(), 1)),
            'presisi': float(lonjakan[teratas].mean()), 'dasar': float(lonjakan.mean())}


def hijriah_ke_masehi(tahun, bulan, hari):
    jd = hari + np.ceil(29.5 * (bulan - 1)) + (tahun - 1) * 354 + (3 + 11 * tahun) // 30 + 1948439.5 - 1
    return pd.Timestamp.fromordinal(int(jd - 2451544.5 + 730120))


def kalender_ramadan(tahun_awal=2006, tahun_akhir=2028):
    bulan_ramadan, bulan_idulfitri = set(), set()
    for th in range(1420, 1455):
        awal, akhir = hijriah_ke_masehi(th, 9, 1), hijriah_ke_masehi(th, 10, 1)
        for hari in pd.date_range(awal, akhir - pd.Timedelta(days=1)):
            bulan_ramadan.add((hari.year, hari.month))
        bulan_idulfitri.add((akhir.year, akhir.month))
    bulan = pd.date_range(f'{tahun_awal}-01-01', f'{tahun_akhir}-12-01', freq='MS')
    return pd.DataFrame({'bulan': bulan,
                         'ramadan_ada': [int((b.year, b.month) in bulan_ramadan) for b in bulan],
                         'idulfitri_ada': [int((b.year, b.month) in bulan_idulfitri) for b in bulan]})


def tambah_fitur_eksternal(out, cuaca=None, oni=None):
    out = out.copy()
    out['_bulan'] = out[KOL_TANGGAL].dt.to_period('M').dt.to_timestamp()
    if cuaca is not None and len(cuaca):
        c = cuaca.copy()
        c['bulan'] = pd.to_datetime(c['bulan'])
        c = c.sort_values([KOL_PASAR, 'bulan']).reset_index(drop=True)
        c['bln'] = c['bulan'].dt.month
        acuan = c[c['bulan'].dt.year <= TAHUN_ACUAN_AKHIR]
        normal = acuan.groupby([KOL_PASAR, 'bln'])[['hujan', 'suhu']].mean().rename(columns={'hujan': 'hujan_normal', 'suhu': 'suhu_normal'})
        c = c.join(normal, on=[KOL_PASAR, 'bln'])
        c['anom_hujan_ini'] = c['hujan'] - c['hujan_normal']
        c['anom_suhu_ini'] = c['suhu'] - c['suhu_normal']
        g = c.groupby(KOL_PASAR)
        c['anom_hujan_lag1'] = g['anom_hujan_ini'].shift(1)
        c['anom_hujan_lag2'] = g['anom_hujan_ini'].shift(2)
        c['anom_suhu_lag1'] = g['anom_suhu_ini'].shift(1)
        c = c.rename(columns={'hujan': 'hujan_ini', 'suhu': 'suhu_ini', 'bulan': '_bulan'})
        kolom = [KOL_PASAR, '_bulan', 'hujan_ini', 'suhu_ini', 'anom_hujan_ini', 'anom_hujan_lag1', 'anom_hujan_lag2', 'anom_suhu_ini', 'anom_suhu_lag1']
        out = out.merge(c[kolom], on=[KOL_PASAR, '_bulan'], how='left')
    if oni is not None and len(oni):
        o = oni.copy()
        o['_bulan'] = pd.to_datetime(o['bulan'])
        o = o.sort_values('_bulan').reset_index(drop=True)
        o['oni_ini'] = o['oni']
        o['oni_lag3'] = o['oni'].shift(3)
        out = out.merge(o[['_bulan', 'oni_ini', 'oni_lag3']], on='_bulan', how='left')
    kal = kalender_ramadan(out['_bulan'].dt.year.min() - 1, out['_bulan'].dt.year.max() + 1).set_index('bulan')
    depan = out['_bulan'] + pd.offsets.MonthBegin(1)
    out['ramadan_ini'] = out['_bulan'].map(kal['ramadan_ada']).values
    out['ramadan_depan'] = depan.map(kal['ramadan_ada']).values
    out['idulfitri_ini'] = out['_bulan'].map(kal['idulfitri_ada']).values
    out['idulfitri_depan'] = depan.map(kal['idulfitri_ada']).values
    return out.drop(columns='_bulan')


def daftar_fitur_eksternal(out):
    return [c for c in out.columns if c.startswith(('hujan_', 'suhu_', 'anom_', 'oni_', 'ramadan_', 'idulfitri_'))]

## Fitur dan target
Fitur dari RTFP: inflasi indeks, inflasi sembilan komoditas, skor kepercayaan, dan rentang harga. Fitur eksternal (bila ada): anomali curah hujan dan suhu NASA POWER, indeks ENSO NOAA, dan penanda Ramadan/Idulfitri. Target: status bulan depan per pasar.

In [ ]:
df = pd.read_csv(data_path, parse_dates=[KOL_TANGGAL])
out = siapkan_fitur(df)
fitur_kaya = daftar_fitur_kaya(out)
cuaca = pd.read_csv(cuaca_path, parse_dates=['bulan']) if os.path.exists(cuaca_path) else None
oni = pd.read_csv(oni_path, parse_dates=['bulan']) if os.path.exists(oni_path) else None
fitur_eksternal = None
if cuaca is not None or oni is not None:
    out = tambah_fitur_eksternal(out, cuaca, oni)
    fitur_eksternal = daftar_fitur_eksternal(out)
    print('Data eksternal dipakai:', fitur_eksternal)
else:
    print('Data eksternal belum ada (jalankan notebook 9); model hanya memakai data RTFP')
dm = siapkan_data_model(out)
print(f'Baris untuk model: {len(dm):,} | jumlah fitur diperkaya: {len(fitur_kaya)}')
pd.Series(dm['kelas_depan']).map(dict(enumerate(NAMA_KELAS))).value_counts().reindex(NAMA_KELAS, fill_value=0)

## Training dan testing berdasarkan waktu

In [ ]:
is_train, tgl_potong = pisah_waktu(dm)
print(f'Training: {is_train.sum():,} baris | Testing mulai {tgl_potong:%b %Y}: {(~is_train).sum():,} baris')

## Bandingkan model (supervised learning)
Target adalah status bulan depan (berlabel), jadi ini klasifikasi terawasi. Model dibandingkan dengan dua baseline: tebakan sederhana (status bulan depan sama dengan bulan ini) dan selalu menebak Normal. Bila data eksternal tersedia, ada satu model tambahan yang memakainya, sehingga sumbangan sumber data kedua dan ketiga bisa dilihat langsung.

In [ ]:
hasil = bandingkan_model(dm, is_train, fitur_kaya, fitur_eksternal)
print('Model terbaik berdasarkan F1 macro:', hasil['terbaik'])
hasil['metrik'].round(3)

In [ ]:
y_test = hasil['y_test']
pred = hasil['pred'][hasil['terbaik']]
print(classification_report(y_test, pred, labels=[0, 1, 2], target_names=NAMA_KELAS, zero_division=0))

## Model sebagai pemeringkat risiko
Selain menebak kelas, model memberi probabilitas yang dipakai untuk mengurutkan pasar. Berikut seberapa banyak lonjakan aktual di data testing yang tertangkap oleh pasar-pasar dengan risiko teratas.

In [ ]:
for persen in (0.05, 0.1, 0.2):
    r = uji_peringkat(dm, is_train, hasil, persen)
    print(f'{persen:.0%} teratas: menangkap {r["tertangkap"]:.0%} lonjakan, presisi {r["presisi"]:.0%}, proporsi lonjakan seluruh data testing {r["dasar"]:.1%}')

## Visualisasi

In [ ]:
os.makedirs(output_dir, exist_ok=True)
cm = confusion_matrix(y_test, pred, labels=[0, 1, 2])
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=NAMA_KELAS).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion matrix - {hasil["terbaik"]}')
ax.set_xlabel('Prediksi')
ax.set_ylabel('Aktual')
plt.savefig(f'{output_dir}/grafik_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
jenis, fitur_terbaik = hasil['skenario'][hasil['terbaik']]
penting = pd.Series(hasil['model'][hasil['terbaik']][-1].feature_importances_, index=fitur_terbaik).sort_values()
penting.plot(kind='barh', figsize=(8, 6), title='Kepentingan fitur pada model terbaik')
plt.savefig(f'{output_dir}/grafik_kepentingan_fitur.png', dpi=150, bbox_inches='tight')
plt.show()

## Prediksi status bulan depan per pasar
Model terbaik dilatih ulang memakai seluruh data lalu dipakai pada bulan terakhir; probabilitas tiap kelas ikut disimpan.

In [ ]:
terbaru, model_final, fitur_final = prediksi_terbaru(out, dm, hasil, fitur_kaya)
print(f'Data bulan terakhir: {terbaru[KOL_TANGGAL].max():%b %Y}')
terbaru['status_prediksi_bulan_depan'].value_counts().reindex(NAMA_KELAS, fill_value=0)

## Simpan hasil

In [ ]:
kolom_simpan = [KOL_PASAR, KOL_TANGGAL, 'inflasi_ini', 'inflasi_lalu'] + [c for c in terbaru.columns if c.startswith('inf_')]
kolom_simpan += [c for c in terbaru.columns if c.startswith('trust_')]
kolom_simpan += [c for c in ['rentang_harga', 'obs_12b', 'interpolasi'] if c in terbaru.columns]
kolom_simpan += ['p_normal', 'p_waspada', 'p_lonjakan', 'status_bulan_ini', 'status_prediksi_bulan_depan']
terbaru[kolom_simpan].to_csv(f'{output_dir}/hasil_klasifikasi_status.csv', index=False)
hasil['metrik'].to_csv(f'{output_dir}/metrik_klasifikasi.csv')
print('Hasil dan metrik tersimpan di', output_dir)